# ⚙️ Project 13: Industrial Equipment Remaining Useful Life (RUL) & Predictive Maintenance
### Industrial IoT, Multi-Sensor Degradation & Asymmetric Failure Forecasting

**Author:** Data Science Portfolio Team  
**Difficulty:** 🟡 Intermediate  
**Domain:** Manufacturing & Industrial IoT  

---
### Notebook Outline:
1. **Environment Setup**
2. **Sensor Telemetry Ingestion & Run-to-Failure Inspection**
3. **Exploratory Data Analysis: Sensor Wear Trajectories**
4. **Rolling Statistics & Piecewise Linear RUL Target Clipping**
5. **Model Benchmarking: Random Forest vs. Gradient Boosting**
6. **Early Warning Alarm System Calibration**

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Industrial IoT environment configured.")

In [ ]:
# Ingestion & Telemetry Inspection
df = pd.read_csv("data/turbofan_sensor_rul.csv")
print(f"Sensor Readings: {len(df)} across {df['engine_id'].nunique()} turbofans.")

plt.figure(figsize=(12, 5))
for eng in [1, 2, 3]:
    subset = df[df['engine_id'] == eng]
    plt.plot(subset['cycle'], subset['sensor_2_temp'], label=f"Engine {eng}")
plt.title("Sensor 2 Temperature Degradation Trajectories", fontweight='bold')
plt.xlabel("Operating Cycle")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.show()

In [ ]:
# RUL Modeling
features = ['cycle', 'sensor_2_temp', 'sensor_3_pressure', 'sensor_4_speed', 'sensor_7_flow']
train_engines = df['engine_id'] <= 30
test_engines = df['engine_id'] > 30

X_train, y_train = df[train_engines][features], df[train_engines]['true_rul_cycles']
X_test, y_test = df[test_engines][features], df[test_engines]['true_rul_cycles']

rf = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

preds = rf.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)

print("=== Turbofan RUL Prediction Evaluation ===")
print(f"Test RMSE: {rmse:.2f} cycles")
print(f"Test MAE:  {mae:.2f} cycles")